<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'> Prompt Engineering
      
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>


<hr style='height:2px;border:none'>


# The job of the prompt

- Instruct the LLM
- Explain where to get more context.
    - MCP Servers
    - Tools
    - Files
- Articulate the process
- Describe the output of the agent
- Describe behaviour when the agent is outside of scope


<hr style='height:2px;border:none'>

<table style="width: 100%; border-collapse: collapse;">
<tr>
<td width="50%" style="padding: 20px; vertical-align: top;">
<h2 style="margin-top: 0;">Structure of a prompt</h2>
<p>While you can just have any text, if you use a consistent structure it is easier to understand and easier to maintain.</p>

<h3>The metadata</h3>
<p>The agent's YAML frontmatter provides discovery information:</p>
<pre style="background-color: #f5f5f5; padding: 15px; border-radius: 6px; overflow-x: auto;"><code>--
name: agent-name
description: what the agent will do
allowed-tools: [ list of tools agent will use ]
argument-hint: [user_task]
model: sonnet
color: blue
--</code></pre>

<h3>Purpose</h3>
<p>A description of the purpose of the agent</p>

<h3>Variables</h3>
<ul style="padding-left: 20px;">
<li>VARIABLE_NAME: $1</li>
<li>VARIABLE2_NAME: constant</li>
</ul>

<h3>Instructions</h3>
<ul style="line-height: 1.8; padding-left: 20px;">
<li>High level instructions</li>
<li>Read in other files for enhanced context @file-name</li>
<li>Leverage key words: <strong>IMPORTANT</strong>, <strong>CRITICAL</strong></li>
<li>What to do with out of scope requests</li>
</ul>

<h3>Workflow</h3>
<ul style="line-height: 1.8; padding-left: 20px;">
<li>Specific steps</li>
<li>Use programmatic logic (e.g. loops, if statements, for statements)</li>
</ul>

<h3>Report</h3>
<ul style="padding-left: 20px;">
<li>Clear description of output of the agent, including sample structure</li>
</ul>
</td>
<td width="50%" style="padding: 20px; text-align: center; vertical-align: top;">
<img src="./images/agentmd.png"
     width="100%"
     max-width="500px"
     alt="AI example"
     style="border: 4px solid #404040; border-radius: 10px;">
</td>
</tr>
</table>

<hr style='height:2px;border:none'>

#### If logic
```
Check to see if the `DATABASE_NAME` is provided. If not, STOP immediately and ask the user to provide it.
```

#### Looping logic
```
- Get a list of tables from the `DATABASE_NAME` database in the Teradata system using base_tableList tool, ignore tables that:
  - called All
- IMPORTANT: For each table, you will follow the `table-loop` below:

<table-loop>
  1. Get all the SQL that has executed against the table in the last `NUM_DAYS` days using the dba_tableSqlList tool
  2. Analyze the returned SQL by cycling through each SQL statement and extract:
     1. Name of the source database and table, save as a tuple using the following format: (source_database.source_table, tardatabase.tartable)
     2. Name of the target database and table, save as a tuple using the following format: (source_database.source_table, tardatabase.tartable)
</table-loop>     
```


<hr style='height:2px;border:none'>

<table style="width: 100%; border-collapse: collapse;">
<tr>
<td width="50%" style="padding: 20px; vertical-align: top;">
<h2 style="margin-top: 0;">Tricks to building good prompts</h2>
<ul style="line-height: 1.8; padding-left: 20px;">
<li><strong>Use the LLM to help build the prompt</strong>
<ul>
<li>Can you help me build an agentic prompt to address…</li>
</ul>
</li>
<li><strong>Ask the LLM to research the domain</strong>
<ul>
<li>We should start by researching: domain, …</li>
</ul>
</li>
<li><strong>Ask the LLM to study any knowledge you already have</strong>
<ul>
<li>You should study @teradata-mcp.md, @teradata-query.md</li>
</ul>
</li>
<li><strong>Ask the LLM to ask any clarifying questions</strong>
<ul>
<li>Ask me as many questions as you need to clarify the requirements</li>
</ul>
</li>
<li><strong>Ask the LLM to write the prompt in a specific output</strong>
<ul>
<li>Please write out the prompt using the following template: @agent-template.md</li>
</ul>
</li>
</ul>
<p style="margin-top: 20px; font-style: italic; color: #666;">Or use the meta prompt provided—it's good for building initial drafts of an agent prompt.</p>
</td>
<td width="50%" style="padding: 20px; text-align: center; vertical-align: top;">
<img src="./images/Progressive-disclosure.png"
     width="100%"
     max-width="500px"
     alt="AI example"
     style="border: 4px solid #404040; border-radius: 10px;">
</td>
</tr>
</table>

<hr style='height:2px;border:none'>
<b style = 'font-size:20px;font-family:Arial'>1. Configure the environment</b>

In [1]:
%%capture
!pip install -U langchain langchain-core langchain-openai panel python-dotenv --quiet


<div class="alert alert-block alert-info">
<p style = 'font-size:16px;font-family:Arial'><b>Please</b><i> restart the kernel after executing the above cell to include/update these libraries into memory for this kernel. The simplest way to restart the Kernel is by typing zero zero: <b> 0 0</b></i> and then clicking <b>Restart</b>.</p>
</div>

<div class="alert alert-block alert-info">
    <p style = 'font-size:16px;font-family:Arial'><i><b>Note:</b> To ensure that the Chatbot interface reflects the latest changes, please reload the page by clicking the <b>Reload</b> or <b>Refresh</b> button or pressing F5 on your keyboard for <b>first-time only</b> This will update the notebook with the latest modifications, and you'll be able to interact with the Chatbot using the new libraries.</i></p></div>

In [2]:
import os
import panel as pn
import param
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

print("Checking if this environment is ready")

if os.path.exists("/home/jovyan/JupyterLabRoot/VantageCloud_Lake/.config/.env"):
    print("Your environment parameter file exist.  Please proceed with this use case.")
    # Load all the variables from the .env file into a dictionary
    env_vars = load_dotenv("/home/jovyan/JupyterLabRoot/VantageCloud_Lake/.config/.env")    
else:
    print("Your environment variables have not been loaded")
    print("Please contact the support team.")

Checking if this environment is ready
Your environment parameter file exist.  Please proceed with this use case.


<hr style='height:2px;border:none'>
<b style = 'font-size:20px;font-family:Arial'>2. Build the chatbot</b>

In [3]:
pn.extension()
env_vars = os.environ
llm_key = env_vars.get("litellm_key")
llm_url = env_vars.get("litellm_base_url")
llm = init_chat_model(
    model="openai-gpt-41",
    model_provider="openai",
    base_url=llm_url,
    api_key=llm_key,
)
class ChatbotApp(param.Parameterized):
    def __init__(self, file_path=None, **params):
        super().__init__(**params)
        self.conversation_history = []

        # Read file context if provided
        self.file_context = ""
        if file_path and os.path.exists(file_path):
            with open(file_path, 'r') as f:
                self.file_context = f.read()
            # Add file context as system message
            system_msg = f"You have access to the following file content:\n\n{self.file_context}"
            self.conversation_history.append(SystemMessage(content=system_msg))

        self.chat_interface = pn.chat.ChatInterface(
            callback=self.respond,
        )

    def respond(self, contents: str, user: str, instance: pn.chat.ChatInterface):
        self.conversation_history.append(HumanMessage(content=contents))
        response = llm.invoke(self.conversation_history)
        assistant_message = response.content
        self.conversation_history.append(AIMessage(content=assistant_message))
        return assistant_message

    def show(self):
        return self.chat_interface


<hr style='height:2px;border:none'>
<b style = 'font-size:20px;font-family:Arial'>3. Working with our meta-agent to build an agent prompt</b>

Cut and paste the following into your chatbot:

``` 
I would like to build an agent to check financial compliance of a company, we have Teradata MCP server with the following tools: 
- tdvs_similarity_search: retrieves the official policy text from the Vector Store
- finance_check_business_loyalty_eligibility : runs the pre-approved SQL eligibility lookup via MCP

You should only use information from the tools, if the tools do not give you information then you should say you do not know.
```


In [5]:
# Create app and pass a file
app = ChatbotApp(file_path="meta-agent-simple.md")
app.show()

ChatInterface(_button_data={'send': _ChatButtonData(i...}, _buttons={'send': Button(align='cen...}, _input_container=Row, _input_layout=Row, _placeholder=ChatMessage, _widgets={'ChatAreaInput': ChatArea...}, callback=<bound method C..., show_button_name=True, sizing_mode='stretch_width', widgets=[ChatAreaInput(css_classes...])